# Fine-Tuning & Customization

You’ll learn: how to adapt pre-trained LLMs for domain-specific use, explore parameter-efficient methods (LoRA), run a quick custom text-generation task, and evaluate fine-tuned outputs.

**Prereqs**: Python, Hugging Face transformers, Colab (GPU strongly recommended).
**Deliverables**: screenshots of training logs, sample generations, and short reflections.

<hr>

**Lab 1 — Quick Domain Adaptation with Prompt Tuning** (20–25 mins)

**Goal**: See how a simple “prompt prefix” can specialize a general model without retraining.

**Setup**


In [ ]:
from transformers import pipeline

gen = pipeline("text-generation", model="distilgpt2")

Device set to use cpu


**Steps**

1. **Baseline**:


In [ ]:
print(
    gen(
        "Summarize: The airplane underwent routine maintenance today.",
        max_new_tokens=40,
    )
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'Summarize: The airplane underwent routine maintenance today.\n\n\nThe plane is scheduled to be returned to the hangar on Wednesday, June 6.\nThis is a new flight, and we are very pleased to send the aircraft back to its flight base'}]


2. **Apply a domain prefix**:


In [ ]:
print(
    gen(
        "As an aviation safety report, summarize: The airplane underwent routine maintenance today.",
        max_new_tokens=40,
    )
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'As an aviation safety report, summarize: The airplane underwent routine maintenance today. It has undergone an inspection for a number of engine defects, including the engine and fuel system. The engine has undergone a repair and maintenance maintenance to ensure that all engines are in compliance with the design and'}]


3. Compare outputs with/without prefix.

4. Try 2 more prefixes (e.g., "As a legal contract…", "As a customer service email…").

**Checkpoint**: screenshots of before/after generations.

**Reflection**: How much did adding context shift style?


In [ ]:
print(
    gen(
        "As a legal contract, summarize: The airplane underwent routine maintenance today.",
        max_new_tokens=40,
    )
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'As a legal contract, summarize: The airplane underwent routine maintenance today.\n\n\nThe problem is that the FAA is failing to comply with the FAA for that reason.\nThe FAA is also failing to comply with the FAA for that reason.\nThe FAA is also'}]


In [5]:
print(
    gen(
        "As a customer service email, summarize: The airplane underwent routine maintenance today.",
        max_new_tokens=40,
    )
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'As a customer service email, summarize: The airplane underwent routine maintenance today. The airplane is returning to service soon.\n\n\n\n\n\n\n\n\n\n\nThe Boeing 777 is returning to service tomorrow.'}]


## Lab 2 — Parameter-Efficient Fine-Tuning with LoRA (30–40 mins)

**Goal**: Use Hugging Face peft to adapt a model with LoRA on a small dataset.

**Setup**


In [ ]:
#!pip install -q transformers datasets peft accelerate bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


**Steps**

1. Load dataset (SST-2 for sentiment, or your own small CSV).


In [7]:
from datasets import load_dataset

ds = load_dataset("sst2")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

2. Load model with peft


In [9]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, get_peft_model

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)
config = LoraConfig(
    r=8, lora_alpha=16, target_modules=["q_lin", "v_lin"], lora_dropout=0.1
)
model = get_peft_model(model, config)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


3. Fine-tune on a subset (few hundred examples).

4. Evaluate accuracy on validation set.

**Checkpoint**: sample training log + accuracy printout.

**Reflection**: Why does LoRA save memory vs full fine-tuning?


## Lab 3 — Full Fine-Tuning on Custom Text (40–50 mins)

**Goal**: Run a small-scale fine-tune of GPT-2 on a custom corpus.

**Setup**


In [ ]:
from datasets import Dataset

texts = [
    "We regret to inform you that your flight has been delayed.",
    "Thank you for flying with us. Your boarding gate is A12.",
    "Weather disruptions may cause schedule changes.",
]
dataset = Dataset.from_dict({"text": texts})

**Steps**

1. Tokenize:


In [ ]:
tok = AutoTokenizer.from_pretrained("distilgpt2")


def encode(ex):
    return tok(ex["text"], truncation=True)


ds = dataset.map(encode)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

2. Use Trainer with AutoModelForCausalLM.

3. Train for 1–2 epochs (tiny dataset = fast).

4. Generate sample output with custom style.

**Checkpoint**: "before" vs "after" generations.

**Reflection**: Did outputs pick up your dataset style? What limitations did you notice?


## Lab 4 — Evaluation & Comparison (25–30 mins)

**Goal**: Compare base vs customized models with simple metrics.

**Steps**

1. Generate text with base GPT-2 and fine-tuned GPT-2.
1. Compute BLEU for similarity to reference (e.g., expected summaries).


In [ ]:
#!pip install -q sacrebleu

import sacrebleu

refs = [["Your flight has been delayed due to weather."]]
sys = ["We regret to inform you that your flight is delayed."]
print(sacrebleu.corpus_bleu(sys, refs).score)

5.300156689756297


3. Manually compare tone/accuracy.

**Checkpoint**: printed BLEU + qualitative notes.

**Reflection**: Which evaluation felt more informative — automated metrics or human inspection?

<hr>

**Grading Rubric** (10 pts)

- Prompt tuning outputs (2 pts)
- LoRA setup + training logs (3 pts)
- Custom fine-tune outputs (3 pts)
- Evaluation reflection (2 pts)
